# Generate AAPL News-Sentiment Features with FinBERT

This notebook converts a large financial-news dataset into the processed sentiment
features used by `garch_sentiment_analysis.ipynb`.

Pipeline:

1. Download the Kaggle financial-headline dataset.
2. Keep headlines from the analysis period that mention Apple.
3. Classify each headline with the pretrained `yiyanghkust/finbert-tone` model.
4. Encode positive, neutral, and negative labels as `1`, `0`, and `-1`.
5. Save article-level results to `data/tech_sentiment_progress.csv`.

The downstream forecasting notebook performs the trading-day alignment and daily
aggregation. Those operations are intentionally not repeated here.

## 1. Imports and configuration

Install the packages in `requirements.txt` before running this notebook. A GPU runtime
is recommended because FinBERT inference over thousands of headlines is computationally
expensive.

In [6]:
from pathlib import Path
import time

import kagglehub
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    pipeline,
)

DATASET_SLUG = "miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests"
MODEL_NAME = "yiyanghkust/finbert-tone"

START_DATE = pd.Timestamp("2010-02-10")
END_DATE = pd.Timestamp("2020-06-04")

BATCH_SIZE = 32
CHECKPOINT_EVERY = 1_000

OUTPUT_PATH = Path("data") / "tech_sentiment_progress.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## 2. Download and load the headline data

In [7]:
dataset_path = Path(kagglehub.dataset_download(DATASET_SLUG))
headline_path = dataset_path / "raw_partner_headlines.csv"

headlines_raw = pd.read_csv(
    headline_path,
    usecols=["date", "headline"],
    parse_dates=["date"]
).dropna(subset=["date", "headline"])

print(f"Loaded {len(headlines_raw):,} headlines from {headline_path.name}.")

Using Colab cache for faster access to the 'massive-stock-news-analysis-db-for-nlpbacktests' dataset.
Loaded 1,845,559 headlines from raw_partner_headlines.csv.


## 3. Select Apple-related headlines

The forecasting target is AAPL, so the raw dataset is restricted to the analysis period
and to headlines containing the keyword `Apple`. This is a transparent keyword-based
filter; entity-level matching would be a possible future improvement.

In [8]:
date_mask = headlines_raw["date"].between(START_DATE, END_DATE)
apple_mask = headlines_raw["headline"].str.contains(
    "apple",
    case=False,
    regex=False,
    na=False
)

apple_headlines = (
    headlines_raw.loc[date_mask & apple_mask, ["date", "headline"]]
    .reset_index(drop=True)
)

print(
    f"Selected {len(apple_headlines):,} Apple-related headlines "
    f"from {len(headlines_raw):,} total headlines."
)

apple_headlines.head()

Selected 12,809 Apple-related headlines from 1,845,559 total headlines.


,date,headline
0,2019-11-12,Royal London Asset Management Ltd Buys Apple I...
1,2019-11-07,"TDAM USA Inc. Buys Phillips 66, DuPont de Nemo..."
2,2018-06-02,"Stocks To Watch: Eyes On Apple's WWDC, SCOTUS ..."
3,2017-11-21,Stocks Up Big As Apple Leads Dow; Is It Time T...
4,2017-05-23,Stocks: Slim Gains Turn Mixed; Nokia Surges On...


## 4. Load pretrained FinBERT

`yiyanghkust/finbert-tone` is a pretrained financial-language model. This project uses
the model for inference; it does not train or fine-tune FinBERT. If a CUDA-compatible
GPU is available, the pipeline uses it automatically.

In [9]:
device = 0 if torch.cuda.is_available() else -1

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

finbert = pipeline(
    task="sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Inference device:", "GPU" if device == 0 else "CPU")

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  439MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inference device: CPU


## 5. Run batched sentiment inference

Headlines are processed in batches of 32. Partial outputs are written approximately
every 1,000 predictions so that a long inference run does not lose all completed work
if the runtime disconnects. A fresh run starts from the beginning and overwrites the
partial output.

In [10]:
label_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

headline_texts = apple_headlines["headline"].tolist()
sentiment_scores = []
next_checkpoint = CHECKPOINT_EVERY


def save_predictions(number_of_predictions):
    partial_results = apple_headlines.iloc[:number_of_predictions].copy()
    partial_results["sentiment"] = sentiment_scores[:number_of_predictions]
    partial_results.to_csv(OUTPUT_PATH, index=False)


for start in tqdm(range(0, len(headline_texts), BATCH_SIZE)):
    batch = headline_texts[start:start + BATCH_SIZE]
    predictions = finbert(batch, truncation=True)

    sentiment_scores.extend(
        label_map[prediction["label"].lower()]
        for prediction in predictions
    )

    if len(sentiment_scores) >= next_checkpoint:
        save_predictions(len(sentiment_scores))
        print(
            f"[{time.strftime('%H:%M:%S')}] "
            f"Saved {len(sentiment_scores):,} predictions to {OUTPUT_PATH}."
        )
        next_checkpoint += CHECKPOINT_EVERY

# Save the complete dataset, including the final partial batch.
save_predictions(len(sentiment_scores))

print(
    f"Completed {len(sentiment_scores):,} predictions and saved the final "
    f"dataset to {OUTPUT_PATH}."
)

  0%|          | 0/401 [00:00<?, ?it/s]

[01:40:23] Saved 1,024 predictions to data/tech_sentiment_progress.csv.
[01:42:18] Saved 2,016 predictions to data/tech_sentiment_progress.csv.
[01:44:38] Saved 3,008 predictions to data/tech_sentiment_progress.csv.
[01:46:50] Saved 4,000 predictions to data/tech_sentiment_progress.csv.
[01:48:53] Saved 5,024 predictions to data/tech_sentiment_progress.csv.
[01:50:54] Saved 6,016 predictions to data/tech_sentiment_progress.csv.
[01:52:54] Saved 7,008 predictions to data/tech_sentiment_progress.csv.
[01:54:47] Saved 8,000 predictions to data/tech_sentiment_progress.csv.
[01:56:42] Saved 9,024 predictions to data/tech_sentiment_progress.csv.
[01:58:34] Saved 10,016 predictions to data/tech_sentiment_progress.csv.
[02:00:26] Saved 11,008 predictions to data/tech_sentiment_progress.csv.
[02:02:25] Saved 12,000 predictions to data/tech_sentiment_progress.csv.
Completed 12,809 predictions and saved the final dataset to data/tech_sentiment_progress.csv.


## 6. Validate the generated feature file

The output contains one row per article. Trading-day alignment and daily averaging are
performed later because they depend on the AAPL market calendar used by the forecasting
experiment.

In [11]:
sentiment_data = pd.read_csv(OUTPUT_PATH, parse_dates=["date"])

print("Output rows:", f"{len(sentiment_data):,}")
print("Missing values:")
print(sentiment_data.isna().sum())
print("\nSentiment distribution:")
print(sentiment_data["sentiment"].value_counts().sort_index())

sentiment_data.head()

Output rows: 12,809
Missing values:
date         0
headline     0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
-1    2018
 0    7811
 1    2980
Name: count, dtype: int64


,date,headline,sentiment
0,2019-11-12,Royal London Asset Management Ltd Buys Apple I...,0
1,2019-11-07,"TDAM USA Inc. Buys Phillips 66, DuPont de Nemo...",0
2,2018-06-02,"Stocks To Watch: Eyes On Apple's WWDC, SCOTUS ...",0
3,2017-11-21,Stocks Up Big As Apple Leads Dow; Is It Time T...,1
4,2017-05-23,Stocks: Slim Gains Turn Mixed; Nokia Surges On...,-1
